# Fundamentals of Machine Learning — Exercise 4

## Hierarchical Clustering, DBSCAN, and Choosing a Clustering Method

In Exercise 3, K-means showed us that clustering depends on representation, distance, scaling, and the chosen number of clusters.

This exercise asks a new question:

> **What should we do when clusters are not compact, spherical groups around a centroid?**

We will compare:

- **K-means**,
- **agglomerative hierarchical clustering**,
- **DBSCAN**.

The classroom workflow remains:

> **question → prediction → computation → visualisation → interpretation → verification**

### Learning outcomes

After completing this exercise, you should be able to:

- explain the bottom-up idea of agglomerative clustering,
- read a dendrogram and explain what a horizontal cut represents,
- distinguish single, complete, and Ward linkage,
- explain the roles of `eps` and `min_samples` in DBSCAN,
- identify core points, border points, and noise,
- compare clustering methods according to data geometry and noise,
- recognise that one numerical metric cannot decide every clustering problem,
- justify a clustering method using several pieces of evidence,
- use an AI chatbot to question a decision after forming your own answer.

**Recommended duration:** 100–120 minutes.  
The final method-selection memo may be completed as homework.

## How to work in this exercise

Most code is already prepared. Your task is to:

1. **predict** the result before running a cell,
2. **change** only clearly marked settings,
3. **compare** alternative methods,
4. **interpret** the output using numerical and visual evidence,
5. **verify** an important claim with a second piece of evidence.

Do not choose a method only because its plot “looks nice”.  
A defensible conclusion should mention:

- the expected cluster shape,
- the role of distance or density,
- sensitivity to noise,
- at least one numerical result,
- one limitation.

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.datasets import make_blobs, make_moons, make_circles, load_wine
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 12

## 1. The same number of groups can have different geometry

We begin with three artificial datasets. Each dataset contains **two known generating groups**, but the shapes are different.

Before running the cell, predict which dataset should be easiest for K-means.

| Dataset | Expected shape | Will K-means work well? Why? |
|---|---|---|
| Compact blobs |  |  |
| Two moons |  |  |
| Concentric circles |  |  |

In [ ]:
X_blobs, y_blobs = make_blobs(
    n_samples=240,
    centers=[(-2.2, -1.7), (2.2, 1.7)],
    cluster_std=[0.65, 0.70],
    random_state=RANDOM_STATE,
)

X_moons, y_moons = make_moons(
    n_samples=240,
    noise=0.07,
    random_state=RANDOM_STATE,
)

X_circles, y_circles = make_circles(
    n_samples=240,
    factor=0.42,
    noise=0.05,
    random_state=RANDOM_STATE,
)

toy_sets = {
    "Compact blobs": (X_blobs, y_blobs),
    "Two moons": (X_moons, y_moons),
    "Concentric circles": (X_circles, y_circles),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for axis, (name, (X, _)) in zip(axes, toy_sets.items()):
    axis.scatter(X[:, 0], X[:, 1], alpha=0.72)
    axis.set_title(name)
    axis.set_xlabel("Feature 1")
    axis.set_ylabel("Feature 2")

plt.tight_layout()
plt.show()

### Run the same K-means model on all three datasets

The value `k=2` is deliberately kept constant. The question is not whether we supplied the correct number of clusters, but whether the **model assumption** fits the geometry.

In [ ]:
kmeans_results = {}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for axis, (name, (X, true_group)) in zip(axes, toy_sets.items()):
    X_scaled = StandardScaler().fit_transform(X)

    labels = KMeans(
        n_clusters=2,
        n_init=20,
        random_state=42,
    ).fit_predict(X_scaled)

    kmeans_results[name] = {
        "labels": labels,
        "ARI against generating groups": adjusted_rand_score(true_group, labels),
        "silhouette": silhouette_score(X_scaled, labels),
    }

    axis.scatter(X[:, 0], X[:, 1], c=labels, cmap="tab10", alpha=0.72)
    axis.set_title(name)
    axis.set_xlabel("Feature 1")
    axis.set_ylabel("Feature 2")

plt.tight_layout()
plt.show()

pd.DataFrame(kmeans_results).T.round(3)

### Activity 1 — Explain a failure, not only a score

Complete the argument:

> K-means performs well on __________ because __________.
>
> K-means fails on __________ even though `k=2`, because __________.
>
> The failure is caused mainly by the method's assumption that __________.

Why is “the algorithm found two clusters” insufficient evidence that the partition is meaningful?

## 2. Agglomerative clustering: building a hierarchy of merges

Agglomerative clustering is a **bottom-up** procedure:

1. each observation starts as its own cluster,
2. the closest pair of clusters is merged,
3. distances between the new clusters are recomputed,
4. merging continues until only one cluster remains.

A dendrogram records this process.

- Leaves at the bottom represent observations.
- A horizontal connection represents a merge.
- The height of a merge represents the dissimilarity at which it occurred.
- A horizontal cut converts the hierarchy into a chosen partition.

In [ ]:
X_small = np.array([
    [-3.2, -2.0], [-2.8, -2.3], [-3.0, -1.6], [-2.6, -1.8],
    [-0.3,  2.4], [ 0.1,  2.1], [ 0.4,  2.5], [ 0.0,  2.8],
    [ 2.7, -1.4], [ 3.1, -1.0], [ 3.4, -1.5], [ 3.0, -1.8],
])
point_names = list("ABCDEFGHIJKL")

plt.figure(figsize=(7, 5))
plt.scatter(X_small[:, 0], X_small[:, 1], s=70)

for name, (x, y) in zip(point_names, X_small):
    plt.text(x + 0.08, y + 0.08, name)

plt.title("Small dataset for reading a dendrogram")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

### Predict before viewing the dendrogram

1. Which points do you expect to merge first?
2. Which three broad groups do you see?
3. At approximately what stage should the algorithm begin merging genuinely different groups?

In [ ]:
linkage_methods = ["single", "complete", "ward"]
small_linkages = {
    method: linkage(X_small, method=method)
    for method in linkage_methods
}

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for axis, method in zip(axes, linkage_methods):
    dendrogram(
        small_linkages[method],
        labels=point_names,
        ax=axis,
    )
    axis.set_title(f"{method.capitalize()} linkage")
    axis.set_xlabel("Observation")
    axis.set_ylabel("Merge distance")

plt.tight_layout()
plt.show()

### Activity 2 — Read the hierarchy

Choose one dendrogram and answer:

1. Name one pair of observations that merges early.
2. Identify one large vertical gap.
3. What number of clusters would a cut inside that gap produce?
4. Why can the answer differ between linkage methods?

Do not write only “three clusters are visible”. Point to a merge height or a gap.

### Change the cut, not the hierarchy

The complete hierarchy remains fixed. We now choose a horizontal cut.

Change only `CUT_HEIGHT` and observe the number of clusters.

In [ ]:
CUT_HEIGHT = 2.0

complete_labels = fcluster(
    small_linkages["complete"],
    t=CUT_HEIGHT,
    criterion="distance",
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

dendrogram(
    small_linkages["complete"],
    labels=point_names,
    ax=axes[0],
)
axes[0].axhline(CUT_HEIGHT, linestyle="--")
axes[0].set_title(f"Complete linkage, cut at {CUT_HEIGHT}")
axes[0].set_ylabel("Merge distance")

axes[1].scatter(
    X_small[:, 0],
    X_small[:, 1],
    c=complete_labels,
    cmap="tab10",
    s=70,
)
for name, (x, y) in zip(point_names, X_small):
    axes[1].text(x + 0.08, y + 0.08, name)
axes[1].set_title(
    f"Partition: {pd.Series(complete_labels).nunique()} clusters"
)
axes[1].set_xlabel("Feature 1")
axes[1].set_ylabel("Feature 2")

plt.tight_layout()
plt.show()

### Stop and explain

Complete the sentences:

> Moving the cut **upward** usually produces __________ clusters because __________.
>
> Moving the cut **downward** usually produces __________ clusters because __________.
>
> Choosing a cut is still a modelling decision because __________.

## 3. Linkage changes what “closest clusters” means

We compare three common choices:

- **Single linkage:** distance between the closest pair of points from two clusters.
- **Complete linkage:** distance between the farthest pair of points.
- **Ward linkage:** merges clusters that cause the smallest increase in within-cluster variance.

Before running the next cell, predict which linkage is most likely to follow the curved moons.

In [ ]:
X_moons_scaled = StandardScaler().fit_transform(X_moons)

moon_linkage_results = {}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for axis, method in zip(axes, linkage_methods):
    labels = AgglomerativeClustering(
        n_clusters=2,
        linkage=method,
    ).fit_predict(X_moons_scaled)

    moon_linkage_results[method] = {
        "silhouette": silhouette_score(X_moons_scaled, labels),
        "ARI against generating groups": adjusted_rand_score(y_moons, labels),
    }

    axis.scatter(
        X_moons[:, 0],
        X_moons[:, 1],
        c=labels,
        cmap="tab10",
        alpha=0.72,
    )
    axis.set_title(f"{method.capitalize()} linkage")
    axis.set_xlabel("Feature 1")
    axis.set_ylabel("Feature 2")

plt.tight_layout()
plt.show()

pd.DataFrame(moon_linkage_results).T.round(3)

### Activity 3 — Link the result to the definition

1. Which linkage best follows the two moons?
2. Explain the result using the definition of that linkage.
3. Which method creates more compact groups?
4. What risk might arise if a few points form a thin “bridge” between two groups?

## 4. DBSCAN: clusters as dense regions

DBSCAN uses two main parameters:

- `eps`: the radius of a neighbourhood,
- `min_samples`: the minimum number of nearby observations needed for a dense region.

It distinguishes:

- **core points** — sufficiently dense neighbourhood,
- **border points** — near a core point but not dense enough themselves,
- **noise** — points that do not belong to a dense region.

DBSCAN does **not** require the number of clusters in advance.  
However, it does require a defensible density scale.

### Predict the effect of `eps`

Keep `min_samples=5`.

| `eps` | Your prediction |
|---:|---|
| 0.10 |  |
| 0.18 |  |
| 0.32 |  |

Use the words **fragment**, **noise**, or **merge** where appropriate.

In [ ]:
EPS_VALUES = [0.10, 0.18, 0.32]
dbscan_rows = []

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for axis, eps in zip(axes, EPS_VALUES):
    labels = DBSCAN(
        eps=eps,
        min_samples=5,
    ).fit_predict(X_moons)

    non_noise = labels != -1
    n_clusters = len(set(labels[non_noise]))

    if n_clusters >= 2 and non_noise.sum() > n_clusters:
        silhouette = silhouette_score(
            X_moons[non_noise],
            labels[non_noise],
        )
    else:
        silhouette = np.nan

    dbscan_rows.append({
        "eps": eps,
        "clusters excluding noise": n_clusters,
        "noise points": int((labels == -1).sum()),
        "silhouette excluding noise": silhouette,
        "ARI against generating groups": adjusted_rand_score(y_moons, labels),
    })

    axis.scatter(
        X_moons[:, 0],
        X_moons[:, 1],
        c=labels,
        cmap="tab10",
        alpha=0.72,
    )
    axis.set_title(f"eps={eps}")
    axis.set_xlabel("Feature 1")
    axis.set_ylabel("Feature 2")

plt.tight_layout()
plt.show()

pd.DataFrame(dbscan_rows).round(3)

### Activity 4 — Explain parameter sensitivity

For each value of `eps`, describe what happened.

Then complete:

> A very small `eps` can create many small clusters or noise because __________.
>
> A very large `eps` can merge distinct groups because __________.
>
> Therefore, DBSCAN does not remove parameter choice; it changes the question from “how many clusters?” to __________.

### A nearest-neighbour distance plot as a diagnostic

For `min_samples=5`, we inspect each observation's distance to its fifth nearest neighbour.

A bend in this sorted curve can suggest an `eps` candidate. It is **not an automatic answer**.

In [ ]:
MIN_SAMPLES = 5

neighbours = NearestNeighbors(n_neighbors=MIN_SAMPLES)
neighbour_distances, _ = neighbours.fit(X_moons).kneighbors(X_moons)

fifth_neighbour_distance = np.sort(
    neighbour_distances[:, -1]
)

plt.figure(figsize=(8, 4.5))
plt.plot(fifth_neighbour_distance)
plt.axhline(0.18, linestyle="--", label="Reference candidate: 0.18")
plt.title("Sorted distance to the 5th nearest neighbour")
plt.xlabel("Observations sorted by distance")
plt.ylabel("Distance")
plt.legend()
plt.show()

### Verification checkpoint

Does the distance plot support considering `eps≈0.18`?

Answer cautiously:

- What pattern do you see?
- Is there one perfectly sharp elbow?
- Why should you still inspect the resulting partition?

### Core, border, and noise points

We now inspect the selected DBSCAN solution directly.

In [ ]:
selected_dbscan = DBSCAN(
    eps=0.18,
    min_samples=5,
).fit(X_moons)

selected_labels = selected_dbscan.labels_

core_mask = np.zeros(len(X_moons), dtype=bool)
core_mask[selected_dbscan.core_sample_indices_] = True

point_role = np.full(len(X_moons), "border", dtype=object)
point_role[core_mask] = "core"
point_role[selected_labels == -1] = "noise"

role_counts = pd.Series(point_role).value_counts()
display(role_counts)

plt.figure(figsize=(8, 5.5))

for role, marker in [("core", "o"), ("border", "s"), ("noise", "x")]:
    mask = point_role == role
    plt.scatter(
        X_moons[mask, 0],
        X_moons[mask, 1],
        c=selected_labels[mask],
        cmap="tab10",
        marker=marker,
        s=65,
        alpha=0.78,
        label=role,
    )

plt.title("DBSCAN point roles")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.show()

### Stop and explain

Why is a border point allowed to belong to a cluster even though its own neighbourhood is not dense enough?

Why is the label `-1` different from an ordinary cluster number?

## 5. Compare methods systematically

We now apply several methods to all three geometries.

For this controlled experiment:

- each dataset is standardised,
- methods that require a cluster count receive `2`,
- DBSCAN uses `eps=0.40` and `min_samples=5`,
- the generating groups are used only as an **external audit**.

In [ ]:
def count_clusters(labels):
    labels = np.asarray(labels)
    return len(set(labels[labels != -1]))

def silhouette_without_noise(X, labels):
    labels = np.asarray(labels)
    mask = labels != -1
    n_clusters = count_clusters(labels)

    if n_clusters < 2 or mask.sum() <= n_clusters:
        return np.nan

    return silhouette_score(X[mask], labels[mask])

comparison_rows = []
comparison_labels = {}

for dataset_name, (X, true_group) in toy_sets.items():
    X_scaled = StandardScaler().fit_transform(X)

    methods = {
        "K-means": KMeans(
            n_clusters=2,
            n_init=20,
            random_state=42,
        ).fit_predict(X_scaled),

        "Agglomerative — single": AgglomerativeClustering(
            n_clusters=2,
            linkage="single",
        ).fit_predict(X_scaled),

        "Agglomerative — complete": AgglomerativeClustering(
            n_clusters=2,
            linkage="complete",
        ).fit_predict(X_scaled),

        "Agglomerative — Ward": AgglomerativeClustering(
            n_clusters=2,
            linkage="ward",
        ).fit_predict(X_scaled),

        "DBSCAN": DBSCAN(
            eps=0.40,
            min_samples=5,
        ).fit_predict(X_scaled),
    }

    for method_name, labels in methods.items():
        comparison_labels[(dataset_name, method_name)] = labels

        comparison_rows.append({
            "dataset": dataset_name,
            "method": method_name,
            "clusters": count_clusters(labels),
            "noise points": int((labels == -1).sum()),
            "silhouette": silhouette_without_noise(X_scaled, labels),
            "ARI against generating groups": adjusted_rand_score(
                true_group,
                labels,
            ),
        })

comparison_table = pd.DataFrame(comparison_rows)
comparison_table.round(3)

In [ ]:
method_order = [
    "K-means",
    "Agglomerative — single",
    "Agglomerative — complete",
    "Agglomerative — Ward",
    "DBSCAN",
]

fig, axes = plt.subplots(
    len(toy_sets),
    len(method_order),
    figsize=(19, 11),
)

for row_index, (dataset_name, (X, _)) in enumerate(toy_sets.items()):
    for column_index, method_name in enumerate(method_order):
        axis = axes[row_index, column_index]
        labels = comparison_labels[(dataset_name, method_name)]

        axis.scatter(
            X[:, 0],
            X[:, 1],
            c=labels,
            cmap="tab10",
            s=25,
            alpha=0.72,
        )

        if row_index == 0:
            axis.set_title(method_name)

        if column_index == 0:
            axis.set_ylabel(dataset_name)

        axis.set_xticks([])
        axis.set_yticks([])

plt.tight_layout()
plt.show()

### Activity 5 — Build an evidence-based recommendation

For each dataset, choose the method you would recommend.

| Dataset | Recommended method | Visual evidence | Numerical evidence | Limitation |
|---|---|---|---|---|
| Compact blobs |  |  |  |  |
| Two moons |  |  |  |  |
| Concentric circles |  |  |  |  |

Then answer:

1. Why can the silhouette score favour a visually incorrect compact partition?
2. Why is ARI available in this experiment but usually unavailable in a real clustering task?
3. What does this comparison teach about the phrase “best clustering algorithm”?

## 6. Real-world example: Wine measurements

We now use the built-in Wine dataset from scikit-learn.

Each row represents a wine sample. The features are chemical measurements.  
The dataset also contains a known cultivar category, but we will **not** use it to create clusters.

The cultivar will be revealed later only as an external audit.

In [ ]:
wine = load_wine(as_frame=True)

wine_features = wine.data.copy()
wine_reference = wine.target.copy()

print("Rows, features:", wine_features.shape)
display(wine_features.head())
display(wine_features.describe().T[["mean", "std", "min", "max"]])

### Predict before preprocessing

1. Why is standardisation necessary here?
2. Which feature would dominate Euclidean distance without scaling?
3. Why should the cultivar label remain outside the clustering matrix?

In [ ]:
wine_scaler = StandardScaler()
X_wine_scaled = wine_scaler.fit_transform(wine_features)

wine_linkage = linkage(
    X_wine_scaled,
    method="ward",
)

plt.figure(figsize=(12, 5))
dendrogram(
    wine_linkage,
    truncate_mode="lastp",
    p=20,
    show_leaf_counts=True,
)
plt.title("Truncated Ward dendrogram — Wine data")
plt.xlabel("Merged subtree")
plt.ylabel("Merge distance")
plt.show()

### Activity 6 — Use the dendrogram as evidence

The dendrogram is truncated because 178 individual leaves would be difficult to read.

Answer:

1. Which candidate counts seem worth testing: 2, 3, 4, or more?
2. What merge-height evidence supports your candidates?
3. What information is hidden by the truncation?

### Compare three candidate methods

To keep the comparison interpretable:

- K-means uses `k=3`,
- Ward agglomerative clustering uses `3` clusters,
- DBSCAN uses a candidate density setting.

The aim is not to force all methods to return the same structure.

In [ ]:
WINE_K = 3
WINE_EPS = 2.30
WINE_MIN_SAMPLES = 3

wine_methods = {
    "K-means": KMeans(
        n_clusters=WINE_K,
        n_init=30,
        random_state=42,
    ).fit_predict(X_wine_scaled),

    "Agglomerative — Ward": AgglomerativeClustering(
        n_clusters=WINE_K,
        linkage="ward",
    ).fit_predict(X_wine_scaled),

    "DBSCAN": DBSCAN(
        eps=WINE_EPS,
        min_samples=WINE_MIN_SAMPLES,
    ).fit_predict(X_wine_scaled),
}

wine_summary_rows = []

for method_name, labels in wine_methods.items():
    wine_summary_rows.append({
        "method": method_name,
        "clusters": count_clusters(labels),
        "noise points": int((labels == -1).sum()),
        "silhouette": silhouette_without_noise(
            X_wine_scaled,
            labels,
        ),
    })

wine_summary = pd.DataFrame(wine_summary_rows)
wine_summary.round(3)

### Before revealing the cultivar

Choose the method you currently prefer for exploratory grouping.

Your initial recommendation:

> I prefer __________ because __________.
>
> My evidence is __________.
>
> One concern is __________.

### Profile the Ward clusters in standardised units

A positive value means the cluster average is above the overall dataset mean.  
A negative value means it is below the mean.

In [ ]:
ward_labels = wine_methods["Agglomerative — Ward"]

wine_scaled_frame = pd.DataFrame(
    X_wine_scaled,
    columns=wine_features.columns,
)
wine_scaled_frame["cluster"] = ward_labels

ward_profile = wine_scaled_frame.groupby("cluster").mean()

plt.figure(figsize=(13, 4.8))
sns.heatmap(
    ward_profile,
    center=0,
    cmap="coolwarm",
    annot=False,
)
plt.title("Ward cluster profiles in standardised units")
plt.xlabel("Chemical feature")
plt.ylabel("Cluster")
plt.show()

ward_profile.round(2)

### Activity 7 — Interpret one real-data cluster

Choose one Ward cluster and write three sentences:

1. identify two or three features with strong deviations,
2. describe the cluster without inventing a causal story,
3. state one important within-cluster detail that the mean profile hides.

**Your interpretation:**

### Guided AI study activity — question your recommendation

Use AI only **after** writing your own recommendation and cluster interpretation.

Paste your draft and use this prompt:

```text
Act as a clustering tutor.
Do not rewrite my answer and do not tell me which method is best.
Ask me exactly three short questions, one at a time:

1. What numerical or visual evidence supports my choice?
2. Do the method's assumptions fit the cluster shape and noise in this dataset?
3. What limitation or alternative method should I mention?

Wait for my answer after each question.
```

Then close the chat and record only:

**One useful question from AI:**  

**One sentence I revised:**  

**Why the revised sentence is more accurate:**  

Do not paste the complete conversation.

### External audit: reveal the cultivar categories

The cultivar was not used during clustering. We now use it only to ask whether the partitions correspond to this known external grouping.

A high ARI is useful evidence for this particular external reference, but it does not automatically prove that the method is best for every analytical purpose.

In [ ]:
wine_external_rows = []

for method_name, labels in wine_methods.items():
    wine_external_rows.append({
        "method": method_name,
        "ARI against cultivar": adjusted_rand_score(
            wine_reference,
            labels,
        ),
    })

wine_external = pd.DataFrame(wine_external_rows)
wine_external.round(3)

### Verification checkpoint

1. Did the external audit support or weaken your initial recommendation?
2. Which conclusion should you revise?
3. Why would it still be incorrect to say that the cultivar labels are the only possible “true” clustering?

## 7. Assessed task — Method-selection memo (2 points)

You are preparing a short recommendation for a colleague who wants exploratory groups in the Wine data.

Use the prepared results and perform **one additional comparison**:

- change `WINE_EPS` to one other defensible value, **or**
- change the agglomerative linkage, **or**
- test one additional cluster count suggested by the dendrogram.

Submit a concise memo containing:

1. **Question:** What grouping decision are you evaluating?
2. **Methods compared:** At least two methods or parameter settings.
3. **Evidence:** One visual and one numerical result.
4. **Recommendation:** A method and parameters.
5. **Cluster interpretation:** A short evidence-based description of one cluster.
6. **Limitation:** One reason the recommendation may not transfer to another dataset or purpose.
7. **AI note:** If AI was used, state one question it asked and one claim you independently verified.

### Marking

| Criterion | Points |
|---|---:|
| Comparison and evidence are technically appropriate | 0.75 |
| Recommendation follows from the evidence | 0.50 |
| Cluster interpretation is cautious and specific | 0.50 |
| Limitation or verified AI note is meaningful | 0.25 |

In [ ]:
# Change ONE clearly marked setting and rerun the comparison.

ALTERNATIVE_EPS = 2.50
ALTERNATIVE_LINKAGE = "complete"
ALTERNATIVE_K = 4

# Option A: alternative DBSCAN
alternative_dbscan = DBSCAN(
    eps=ALTERNATIVE_EPS,
    min_samples=WINE_MIN_SAMPLES,
).fit_predict(X_wine_scaled)

# Option B: alternative agglomerative linkage
alternative_agglomerative = AgglomerativeClustering(
    n_clusters=ALTERNATIVE_K,
    linkage=ALTERNATIVE_LINKAGE,
).fit_predict(X_wine_scaled)

additional_comparison = pd.DataFrame([
    {
        "setting": f"DBSCAN eps={ALTERNATIVE_EPS}",
        "clusters": count_clusters(alternative_dbscan),
        "noise points": int((alternative_dbscan == -1).sum()),
        "silhouette": silhouette_without_noise(
            X_wine_scaled,
            alternative_dbscan,
        ),
        "ARI against cultivar": adjusted_rand_score(
            wine_reference,
            alternative_dbscan,
        ),
    },
    {
        "setting": (
            f"Agglomerative {ALTERNATIVE_LINKAGE}, "
            f"k={ALTERNATIVE_K}"
        ),
        "clusters": count_clusters(alternative_agglomerative),
        "noise points": int((alternative_agglomerative == -1).sum()),
        "silhouette": silhouette_without_noise(
            X_wine_scaled,
            alternative_agglomerative,
        ),
        "ARI against cultivar": adjusted_rand_score(
            wine_reference,
            alternative_agglomerative,
        ),
    },
])

additional_comparison.round(3)

### Your method-selection memo

**Question:**  

**Methods and settings compared:**  

**Visual evidence:**  

**Numerical evidence:**  

**Recommendation:**  

**Interpretation of one cluster:**  

**Limitation:**  

**Optional verified AI note:**

# End-of-lab self-check

Before finishing, check that you can:

- [ ] explain how agglomerative clustering builds a hierarchy,
- [ ] read merge height on a dendrogram,
- [ ] explain what changes when a dendrogram cut moves,
- [ ] distinguish single, complete, and Ward linkage,
- [ ] explain `eps` and `min_samples`,
- [ ] distinguish core, border, and noise points,
- [ ] identify fragmentation and merging in DBSCAN,
- [ ] explain why silhouette alone may be misleading,
- [ ] recommend a method based on geometry, density, noise, and evidence,
- [ ] revise an interpretation after questions without copying an AI answer.

Exercise 5 will move from unsupervised learning to optimisation and linear models.